# 07 — Misclassification analysis (Figure 7)

**Tujuan.** Siapkan metadata kesalahan dari final-test JSON yang sudah dibekukan, tanpa inference ulang. Figure 7 hlm. 91512–91513 menunjukkan DDR DR→No DR (FN) dan No DR→DR (FP); ini semantik biner, berbeda dari salah grade lima kelas. Sel readiness **tidak membuka** gambar test. Panel statis kesalahan yang sudah ditransfer dibahas terpisah secara retrospektif.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.visualization import load_controlled_result, error_records
if ARTIFACT_ROOT:
    item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", "ddr_full")
    print("DDR", item["status"])
    if item["status"] == "AVAILABLE":
        errors = error_records(item["result"])
        print(errors["status"], "error records", len(errors.get("errors", [])),
              "task", errors.get("task"))
        binary = error_records(item["result"], binary_projection=True)
        print("binary DR/No DR projection (ASSUMED)", binary["status"],
              "FP/FN records", len(binary.get("errors", [])))
        print("Gallery PENDING: explicit post-evaluation image opt-in required")
else:
    print("PENDING: frozen per-image prediction metadata unavailable")

PENDING: frozen per-image prediction metadata unavailable


In [3]:
from reproduction.cadvit.visualization import plot_error_gallery, save_campaign_figure
if os.environ.get("CAD_VIT_SHOW_TEST_GALLERY") == "1":
    from IPython.display import display
    if ARTIFACT_ROOT is None:
        raise ValueError("CAD_VIT_ARTIFACT_ROOT required")
    item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", "ddr_full")
    if item["status"] != "AVAILABLE":
        raise ValueError("Frozen DDR final-test JSON unavailable or unverified")
    rows = error_records(item["result"], binary_projection=True)
    if rows["status"] != "AVAILABLE":
        raise ValueError("Per-image prediction metadata unavailable")
    dataset_root = os.environ["CAD_VIT_DATA_ROOT"]
    figure = plot_error_gallery(rows["errors"], dataset_root, allow_test_images=True)
    display(figure)
    if os.environ.get("CAD_VIT_SAVE_FIGURES") == "1":
        print("Saved", save_campaign_figure(figure, ARTIFACT_ROOT,
                                             "balanced_v1", "ddr_full", "error_gallery"))
else:
    print("PENDING gallery: test images remain unopened; opt in only after final evaluation is frozen")

PENDING gallery: test images remain unopened; opt in only after final evaluation is frozen


**Interpretasi dan keterbatasan.** Runner baru menyimpan image ID, label aktual/prediksi, confidence, serta SHA-256 di hasil final test **pada run yang memang menyimpan prediksi per gambar**. `plot_error_gallery(..., allow_test_images=True)` tersedia untuk tahap pelaporan setelah evaluasi dikunci; jangan memilih contoh berdasarkan test untuk training/tuning. JSON historis tanpa prediksi per gambar tetap PENDING.
**Implementation assumption:** Galeri FP/FN memproyeksikan grade 0 ke No DR dan grade 1–4 ke DR. Kesalahan grade lima kelas tetap dilaporkan terpisah; proyeksi ini tidak diklaim sebagai protokol Figure 7 penulis.


**DDR recovery 80 epoch:** ringkasan final test dan confusion matrix agregat kini tersedia pada bagian berikut. Prediksi lengkap per gambar tetap berada di server; transfer terbaru menyediakan sembilan contoh terpilih dan metadatanya untuk pelaporan retrospektif di bawah. Sel galeri interaktif di atas mencari root kampanye `balanced_v1` dan tidak digunakan untuk menghasilkan panel transfer.

## Pola kesalahan agregat enam final test

Jumlah salah = N − jumlah diagonal confusion matrix. Pasangan diurutkan menurut frekuensi off-diagonal; arah `true→predicted`. Ini mengidentifikasi pola pengelompokan label, **bukan penyebab klinis**. Sumber V1 adalah matriks historis dalam arsip; empat hasil lain berasal dari JSON final test beku.

| Campaign | Salah / N | Tiga pasangan terbesar (true→pred) | Sumber |
| --- | ---: | --- | --- |
| V1 APTOS | 101/366 | 2→3: 26, 2→1: 16, 4→3: 13 | [matriks](../experiments/01_experiment_initial_cadvit_v1.ipynb) |
| V1 DDR | 712/1252 | 0→1: 238, 2→1: 124, 2→0: 79 | [matriks](../experiments/01_experiment_initial_cadvit_v1.ipynb) |
| V2 BG10 APTOS | 77/366 | 2→1: 16, 3→4: 8, 1→0: 8 | [matriks](../../results/APTOS2019/v2_bg10_historical_80epochs/final_test_v2_80ep.json) |
| V2 BG10 DDR | 245/1252 | 2→0: 51, 2→1: 38, 0→2: 25 | [matriks](../../results/DDR/v2_bg10_historical_80epochs/final_test_v2_80ep.json) |
| Balanced APTOS | 98/366 | 2→3: 14, 2→1: 13, 1→2: 13 | [matriks](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/final_test_summary.json) |
| Recovery DDR | 277/1252 | 0→1: 74, 2→1: 52, 2→0: 41 | [matriks](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json) |

Pola kelas lima grade tidak sama dengan Figure 7 paper yang membahas DR/No DR. Proyeksi biner grade 0→No DR dan grade 1–4→DR adalah pilihan analisis tambahan; salah grade 2→3 tetap salah dalam lima kelas, tetapi tidak menjadi FP/FN biner. Karena hanya matriks agregat yang tersedia, gambar yang salah, keyakinan model, kualitas citra, dan lesi tidak dapat diaudit lokal. Galeri test hanya dapat dibuat retrospektif dari prediksi per gambar yang sudah tersimpan di server, setelah izin publikasi dan provenance gambar diverifikasi. Final-test inference tidak perlu dan tidak boleh diulang untuk memenuhi dokumentasi ini.

## Galeri kesalahan DDR recovery — pelaporan retrospektif

![DDR: sembilan contoh kesalahan dari prediksi final test yang tersimpan](../report_figures/DDR_error_gallery.png)

Judul “Historical” yang tercetak pada PNG menandai galeri arsip retrospektif; hash hasil pada metadatanya mengaitkan panel DDR ini dengan **recovery final test**, bukan run V1/V2. Sumber: [metadata per gambar](../report_figures/DDR_error_metadata.json) dalam ZIP transfer berhash cocok, dengan `result_sha256=4575f79eb536999c3ee0a9cdfb4a52dd01cf613020a1174556563df2367e1ea3`, sama dengan hash raw `test_metrics.json` yang tercatat dalam [provenance evaluator](../provenance/ddr_final_test/README.md). Kategori FP/FN di sini memakai proyeksi biner **grade 0 = No DR, grade 1–4 = DR**; “severity error” salah grade tetapi tetap dalam kelompok DR. Direktori `train/valid/test` di `image_path` ialah struktur **dataset DDR asli**, sedangkan `split=test` metadata merujuk split reproduksi beku. Keduanya tidak boleh disamakan. Hash manifest tercatat, tetapi file manifest dan citra sumber belum ada lokal untuk memverifikasi ulang setiap entri.

| Image ID | Kategori metadata | True→pred grade | Confidence prediksi |
| --- | --- | ---: | ---: |
| 20170406165101998.jpg | False Positive | 0→1 | 0.980 |
| 007-7530-601.jpg | False Positive | 0→1 | 0.915 |
| 20170228164926158.jpg | False Positive | 0→1 | 0.847 |
| 007-3335-200.jpg | False Negative | 2→0 | 0.756 |
| 007-2004-100.jpg | False Negative | 2→0 | 0.956 |
| 20170616090802344.jpg | False Negative | 2→0 | 0.649 |
| 20170514222246265.jpg | Severity Error | 2→1 | 0.988 |
| 007-4198-200.jpg | Severity Error | 2→3 | 0.772 |
| 20170605163201114.jpg | Severity Error | 2→1 | 0.536 |

**Pengamatan gambar.** Tiga FP adalah grade 0 yang diprediksi grade 1; tampilannya bervariasi, termasuk satu fundus gelap dan satu lebih pucat, sehingga contoh ini tidak mendukung satu penjelasan visual tunggal. Tiga FN adalah grade 2 yang diprediksi grade 0; dua panel mempunyai pencahayaan tidak merata dan satu menampilkan area terang dekat diskus. Tiga kesalahan severity berasal dari grade 2 dengan prediksi 1 atau 3; perbedaan kecerahan dan detail kecil tampak antargambar. Tidak ada anotasi lesi atau audit klinis di paket, sehingga ciri yang terlihat tidak dinyatakan sebagai sebab keputusan model.

Kesembilan contoh dipilih dari prediksi final test yang **sudah disimpan**, bukan sampel acak untuk mengestimasi frekuensi kesalahan. Jumlah kesalahan representatif tetap berasal dari confusion matrix 1.252 gambar di atas. Galeri tidak dipakai untuk memilih checkpoint, mengubah protokol, atau merancang novelty. Sumber OIA-DDR dan lisensi panel turunan: [CC BY-NC-SA 4.0 dan atribusi](../report_figures/README.md#ddr-real-image-visualization-transfer).

## Galeri kesalahan APTOS2019 — lokal, retrospektif

![APTOS2019 lokal: sembilan kesalahan dari prediksi final test tersimpan](../report_figures/APTOS2019_error_gallery.png)

[Metadata per gambar](../report_figures/APTOS2019_error_metadata.json) mencatat `split=test`, manifest SHA-256 `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854`, dan `result_sha256=3a987907c3e450cfbc1e6b52546cc17de580b7be36112576468f31f10edc9e57`. Hash raw result dan hash setiap citra berasal dari metadata server; file raw result, manifest, dan citra sumber belum tersedia lokal untuk verifikasi ulang. `train_images/` pada path metadata adalah nama direktori APTOS asli, sedangkan `split=test` adalah keanggotaan split reproduksi beku. Kategori FP/FN memakai proyeksi grade 0 = No DR, grade 1–4 = DR; *severity error* berarti salah grade dalam kelompok DR.

| Image ID | Kategori metadata | True→pred grade | Confidence prediksi |
| --- | --- | ---: | ---: |
| `a85cda5f725d` | False Positive | 0→3 | 0,492 |
| `6a2c3f4ef329` | False Positive | 0→1 | 0,877 |
| `5078caaf1f57` | False Positive | 0→1 | 0,559 |
| `e12b67835e03` | False Negative | 2→0 | 0,960 |
| `d5b4705ac2ee` | False Negative | 1→0 | 0,894 |
| `0124dffecf29` | False Negative | 1→0 | 0,498 |
| `c1896142a20a` | Severity Error | 4→2 | 0,616 |
| `a3802934bad7` | Severity Error | 2→1 | 0,510 |
| `8000a6b97a84` | Severity Error | 2→4 | 0,442 |

**Pengamatan gambar.** Tiga FP grade 0→1/3 memiliki variasi kecerahan dan warna; dua FN 1→0 serta FN 2→0 mencakup confidence tinggi, termasuk **0,960** pada grade 2→0. Tiga kesalahan severity mencakup dua arah dari grade 2 dan satu dari grade 4. Panel menampilkan detail terang dan gelap yang berbeda antargambar, tanpa anotasi lesi atau penilaian klinis independen; dari gambar ini tidak dapat ditetapkan penyebab kesalahan model. Sembilan contoh dipilih retrospektif dari hasil test tersimpan dan **tidak mengestimasi prevalensi** kategori kesalahan. Metrik agregat APTOS tetap berasal dari [final-test summary](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/final_test_summary.json), bukan dari galeri.

File APTOS disimpan lokal dan diabaikan Git karena izin redistribusi belum dikonfirmasi. Gambar terlihat tanpa kernel di salinan lokal ini, tetapi tidak ada di GitHub/clone baru. [Status transfer dan hak berbagi](../report_figures/README.md#aptos2019-local-only-visualization-transfer).

## DDR recovery 80 epoch — analisis kesalahan agregat final test

`DDR_FINAL_TEST` mencatat **277** kesalahan dari **1.252** gambar, berdasarkan [`final_test_summary.json`](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json). Grade **1** mempunyai F1 terendah (**0,323144**). Grade **3** memiliki support **23** dan hanya **11** prediksi benar (recall **0,478261**), sehingga estimasinya tidak pasti. Kebingungan grade **2→3** berjumlah **16**, sedangkan **3→2** berjumlah **11**. Kesalahan paling banyak secara absolut adalah **0→1: 74** dan **2→1: 52**. Matriks agregat tidak menjelaskan penyebab klinis atau ciri visual kesalahan tersebut.

Sel read-only berikut menghitung ulang jumlah kesalahan dan pasangan grade dari matriks asli. Prediksi lengkap per gambar tetap pada server GPU; sembilan contoh terpilih kini tersedia sebagai panel statis terpisah di bagian sebelumnya. Sel galeri `balanced_v1` historis di atas tetap terpisah dan memerlukan opt-in tersendiri.

In [4]:
# Aggregate-only DDR recovery error analysis; no test images or inference.
import json
from pathlib import Path
summary_path = ROOT / "results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json"
if not summary_path.is_file():
    print("PENDING — DDR final-test summary is unavailable")
else:
    result = json.loads(summary_path.read_text(encoding="utf-8"))
    cm = result["confusion_matrix"]
    if (result.get("dataset") != "DDR" or result.get("split") != "test"
            or result.get("campaign") != "cadvit_tables_1_5_balanced_fp32_math_mb8_diag"
            or len(cm) != 5 or any(len(row) != 5 for row in cm)
            or sum(map(sum, cm)) != result["test_samples"]):
        raise ValueError("DDR final-test matrix or provenance is inconsistent")
    errors = sorted(((cm[i][j], i, j) for i in range(5) for j in range(5)
                     if i != j and cm[i][j]), reverse=True)
    print("DDR_FINAL_TEST; errors:", sum(count for count, _, _ in errors))
    print("Largest confusions (count, true, predicted):", errors[:5])
    for item in result["per_class"]:
        print("Grade", item["grade"], "F1", item["f1"], "recall", item["recall"],
              "support", item["support"])
    print("Full per-image predictions remain on the server; selected static examples are linked above")